# Systematic Review Automation — Google Colab
**LLMs in Qualitative Data Analysis (2023–2026)**

Run cells top to bottom. Cell 5 gives you a public URL — no accounts needed.

---
### Before you start — add your OpenAI API key as a Colab Secret
1. Click the **🔑 key icon** in the left sidebar → **+ Add new secret**
2. Name: `OPENAI_API_KEY` · Value: your `sk-...` key · Toggle **Notebook access** ON

Your key is encrypted in your Google account — never visible in output, never sent anywhere.


## Cell 1 — Clone repo + install dependencies

In [ ]:
import os, sys

REPO_URL = 'https://github.com/qbzdm2024/Zhihong.github.io.git'
BRANCH   = 'claude/systematic-review-automation-oC3BZ'
REPO_DIR = '/content/sysrev'
APP_DIR  = f'{REPO_DIR}/systematic-review'

if not os.path.exists(REPO_DIR):
    !git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}
else:
    print('Repo already present — pulling latest...')
    !git -C {REPO_DIR} pull origin {BRANCH}

%cd {APP_DIR}
print('Working directory:', os.getcwd())

In [ ]:
%%capture
!pip install \
    "fastapi==0.115.0" "uvicorn[standard]" \
    "pydantic>=2.7.0" "pydantic-settings>=2.3.0" \
    "openai>=1.50.0" "rispy>=0.9.0" \
    "python-multipart" "python-dotenv" "PyMuPDF" \
    "requests" "xmltodict"

In [ ]:
import fastapi, openai, pydantic, requests
try:
    import rispy
    rispy_ver = rispy.__version__
except ImportError:
    rispy_ver = 'NOT INSTALLED — re-run the pip cell above'

print(f'fastapi {fastapi.__version__} | openai {openai.__version__} | pydantic {pydantic.__version__}')
print(f'rispy   {rispy_ver}')

for d in ['data/raw','data/deduped','data/screened','data/extracted','data/output','data/pdfs']:
    os.makedirs(d, exist_ok=True)
print('Data directories ready.')

## Cell 2 — Load API key from Colab Secrets

In [ ]:
from google.colab import userdata

try:
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
except Exception:
    raise RuntimeError(
        'OPENAI_API_KEY not found in Colab Secrets.\n'
        'Click the 🔑 key icon → Add new secret → Name: OPENAI_API_KEY'
    )

if not OPENAI_API_KEY or not OPENAI_API_KEY.startswith('sk-'):
    raise ValueError('Key must start with sk-')

with open('.env', 'w') as f:
    f.write(f'OPENAI_API_KEY={OPENAI_API_KEY}\n')
    f.write('MODEL_TITLE_SCREENING=gpt-4o-mini\n')
    f.write('MODEL_FULLTEXT_SCREENING=gpt-4o\n')
    f.write('MODEL_EXTRACTION=gpt-4o\n')
    f.write('MODEL_QA_ASSESSMENT=gpt-4o\n')
    f.write('MODEL_AGENT2_SCREENING=gpt-4o\n')
    f.write('MODEL_AGENT2_EXTRACTION=gpt-4o-mini\n')
    f.write('CONFIDENCE_THRESHOLD=0.80\n')

print(f'✓ API key loaded (sk-...{OPENAI_API_KEY[-4:]})')
print('✓ .env written — setup wizard will be skipped')

## Cell 3a — Automated database search (free, no login required)

Searches **PubMed, arXiv, OpenAlex, and Semantic Scholar** automatically.
All results are saved to `data/raw/` ready for import.

These four sources are free and open — **no institutional access needed.**
For Scopus, Web of Science, PsycINFO, ACM, and IEEE, see Cell 3b below.

In [ ]:
import requests, json, time, xml.etree.ElementTree as ET
from urllib.parse import quote_plus
from datetime import datetime

os.makedirs('data/raw', exist_ok=True)

# ── Search configuration ────────────────────────────────
DATE_FROM = '2023'
DATE_TO   = '2026'
MAX_RESULTS_PER_SOURCE = 500   # increase if you want more

# ── Core search terms (from refined protocol) ───────────
LLM_TERMS = [
    'large language model', 'large language models', 'LLM', 'LLMs',
    'generative AI', 'generative artificial intelligence',
    'GPT-4', 'GPT-3', 'GPT-3.5', 'ChatGPT', 'GPT',
    'Claude', 'Llama', 'Llama 2', 'Llama 3',
    'Gemini', 'Mistral', 'PaLM', 'foundation model',
]
QDA_TERMS = [
    'qualitative analysis', 'qualitative research', 'qualitative data analysis',
    'thematic analysis', 'content analysis', 'grounded theory',
    'framework analysis', 'narrative analysis', 'discourse analysis',
    'inductive coding', 'deductive coding', 'open coding',
    'codebook', 'qualitative coding', 'interpretive phenomenological',
]

print(f'Search scope: {DATE_FROM}–{DATE_TO} | max {MAX_RESULTS_PER_SOURCE} per source')
print(f'LLM terms: {len(LLM_TERMS)} | QDA terms: {len(QDA_TERMS)}')

In [ ]:
# ════════════════════════════════════════════════════════
# PubMed  (NCBI E-utilities — completely free, no key)
# Uses your refined Title/Abstract-restricted strategy
# ════════════════════════════════════════════════════════
print('Searching PubMed with refined strategy...')

EUTILS = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils'

# ── Refined strategy: Title/Abstract field restrictions ──────────────────
# Matches the strategy you manually executed (saved in docs/search_strategies.md)
LLM_TIAB = (
    '"large language model"[Title/Abstract] OR "large language models"[Title/Abstract] '
    'OR ChatGPT[Title/Abstract] OR GPT-4[Title/Abstract] OR "GPT 4"[Title/Abstract] '
    'OR Claude[Title/Abstract] OR Llama[Title/Abstract] OR Gemini[Title/Abstract]'
)
QDA_TIAB = (
    '"qualitative analysis"[Title/Abstract] OR "qualitative data analysis"[Title/Abstract] '
    'OR "thematic analysis"[Title/Abstract] OR "content analysis"[Title/Abstract] '
    'OR "grounded theory"[Title/Abstract] OR coding[Title/Abstract] '
    'OR codebook[Title/Abstract] OR "theme development"[Title/Abstract] '
    'OR "automated coding"[Title/Abstract]'
)

pubmed_query = (
    f'({LLM_TIAB}) AND ({QDA_TIAB}) '
    f'AND ("{DATE_FROM}/01/01"[Date - Publication] : "{DATE_TO}/12/31"[Date - Publication]) '
    f'AND (english[Language])'
)

print(f'  Query length: {len(pubmed_query)} chars')

# Step 1: get IDs
search_r = requests.get(f'{EUTILS}/esearch.fcgi', params={
    'db': 'pubmed', 'term': pubmed_query,
    'retmax': MAX_RESULTS_PER_SOURCE, 'retmode': 'json',
    'usehistory': 'y',
}, timeout=30)
search_data = search_r.json()
count  = int(search_data['esearchresult']['count'])
webenv = search_data['esearchresult'].get('webenv', '')
qkey   = search_data['esearchresult'].get('querykey', '')
ids    = search_data['esearchresult']['idlist']
print(f'  Total matching: {count} | Fetching: {len(ids)}')

# Step 2: fetch XML records
if ids:
    time.sleep(0.4)  # NCBI rate limit: 3 req/s
    fetch_r = requests.get(f'{EUTILS}/efetch.fcgi', params={
        'db': 'pubmed', 'query_key': qkey, 'WebEnv': webenv,
        'rettype': 'xml', 'retmode': 'xml',
        'retmax': MAX_RESULTS_PER_SOURCE,
    }, timeout=60)
    out_path = 'data/raw/pubmed_results.xml'
    with open(out_path, 'wb') as f:
        f.write(fetch_r.content)
    print(f'  ✓ Saved {len(ids)} records → {out_path}')
    print(f'  Note: If you manually exported from PubMed, your .nbib file takes precedence')
    print(f'        (both will be imported; duplicates are removed automatically)')
else:
    print('  No PubMed results found.')

time.sleep(0.4)

In [ ]:
# ════════════════════════════════════════════════════════
# arXiv  (free API, no key — cs.AI, cs.CL, cs.HC)
# ════════════════════════════════════════════════════════
print('Searching arXiv...')

import xml.etree.ElementTree as ET

llm_arxiv = ' OR '.join(f'abs:"{t}"' for t in LLM_TERMS[:8])  # keep query short
qda_arxiv = ' OR '.join(f'abs:"{t}"' for t in QDA_TERMS[:8])
arxiv_query = f'({llm_arxiv}) AND ({qda_arxiv})'

arxiv_records = []
batch_size = 100
start = 0
import uuid

while start < MAX_RESULTS_PER_SOURCE:
    r = requests.get('http://export.arxiv.org/api/query', params={
        'search_query': arxiv_query,
        'start': start,
        'max_results': batch_size,
        'sortBy': 'submittedDate',
        'sortOrder': 'descending',
    }, timeout=30)

    ns = {'atom': 'http://www.w3.org/2005/Atom'}
    root = ET.fromstring(r.content)
    entries = root.findall('atom:entry', ns)
    if not entries:
        break

    for entry in entries:
        published = entry.findtext('atom:published', '', ns)[:4]
        if published < DATE_FROM or published > DATE_TO:
            continue
        arxiv_records.append({
            'record_id': str(uuid.uuid4()),
            'source_db': 'arXiv',
            'title': (entry.findtext('atom:title', '', ns) or '').strip().replace('\n', ' '),
            'authors': '; '.join(
                (a.findtext('atom:name', '', ns) or '')
                for a in entry.findall('atom:author', ns)
            ),
            'year': int(published) if published.isdigit() else None,
            'journal_venue': 'arXiv preprint',
            'abstract': (entry.findtext('atom:summary', '', ns) or '').strip().replace('\n', ' '),
            'doi': '',
            'url': entry.findtext('atom:id', '', ns),
        })

    start += batch_size
    if len(entries) < batch_size:
        break
    time.sleep(3)  # arXiv asks for 3s between requests

if arxiv_records:
    out_path = 'data/raw/arxiv_results.json'
    with open(out_path, 'w') as f:
        json.dump(arxiv_records, f, indent=2)
    print(f'  ✓ Saved {len(arxiv_records)} records → {out_path}')
else:
    print('  No arXiv results in date range.')

In [ ]:
# ════════════════════════════════════════════════════════
# OpenAlex  (free, open — covers Scopus/WoS/PubMed)
# https://openalex.org — no API key needed for <100k req/day
# ════════════════════════════════════════════════════════
print('Searching OpenAlex...')

# Build search query — OpenAlex uses simple keyword search
llm_kw  = ' OR '.join(f'"{t}"' for t in [
    'large language model', 'ChatGPT', 'GPT-4', 'LLM', 'Llama', 'Gemini', 'generative AI'
])
qda_kw  = ' OR '.join(f'"{t}"' for t in [
    'qualitative analysis', 'thematic analysis', 'content analysis',
    'grounded theory', 'qualitative research', 'qualitative coding',
])

openalex_records = []
page = 1
per_page = 100
EMAIL = 'sysrev@example.com'  # polite pool (faster responses)

while len(openalex_records) < MAX_RESULTS_PER_SOURCE:
    r = requests.get('https://api.openalex.org/works', params={
        'search': f'({llm_kw}) AND ({qda_kw})',
        'filter': f'publication_year:{DATE_FROM}-{DATE_TO},language:en',
        'per-page': per_page,
        'page': page,
        'mailto': EMAIL,
    }, timeout=30)

    if r.status_code != 200:
        print(f'  OpenAlex error {r.status_code}: {r.text[:200]}')
        break

    data = r.json()
    results = data.get('results', [])
    if not results:
        break

    for w in results:
        authors = '; '.join(
            a.get('author', {}).get('display_name', '')
            for a in (w.get('authorships') or [])[:6]
        )
        abstract = ''
        if w.get('abstract_inverted_index'):
            # Reconstruct abstract from inverted index
            idx = w['abstract_inverted_index']
            words = {pos: word for word, positions in idx.items() for pos in positions}
            abstract = ' '.join(words[i] for i in sorted(words))

        doi = w.get('doi') or ''
        if doi.startswith('https://doi.org/'):
            doi = doi[len('https://doi.org/'):]

        venue = ''
        if w.get('primary_location') and w['primary_location'].get('source'):
            venue = w['primary_location']['source'].get('display_name', '')

        openalex_records.append({
            'record_id': str(uuid.uuid4()),
            'source_db': 'OpenAlex',
            'title': w.get('title') or '',
            'authors': authors,
            'year': w.get('publication_year'),
            'journal_venue': venue,
            'abstract': abstract,
            'doi': doi,
            'url': w.get('id') or '',
        })

    total = data.get('meta', {}).get('count', 0)
    print(f'  Page {page}: {len(results)} results (total available: {total})')

    if page * per_page >= min(total, MAX_RESULTS_PER_SOURCE):
        break
    page += 1
    time.sleep(1)

if openalex_records:
    out_path = 'data/raw/openalex_results.json'
    with open(out_path, 'w') as f:
        json.dump(openalex_records, f, indent=2)
    print(f'  ✓ Saved {len(openalex_records)} records → {out_path}')
else:
    print('  No OpenAlex results.')

In [ ]:
# ════════════════════════════════════════════════════════
# Semantic Scholar  (free API — strong CS/AI coverage)
# https://api.semanticscholar.org
# ════════════════════════════════════════════════════════
print('Searching Semantic Scholar...')

s2_records = []
offset = 0
limit = 100
query = 'large language model qualitative analysis thematic analysis coding'

while len(s2_records) < MAX_RESULTS_PER_SOURCE:
    r = requests.get('https://api.semanticscholar.org/graph/v1/paper/search', params={
        'query': query,
        'fields': 'title,authors,year,abstract,externalIds,venue,publicationDate',
        'limit': limit,
        'offset': offset,
    }, timeout=30)

    if r.status_code == 429:
        print('  Rate limited — waiting 10s...')
        time.sleep(10)
        continue

    if r.status_code != 200:
        print(f'  S2 error {r.status_code}')
        break

    data = r.json()
    papers = data.get('data', [])
    if not papers:
        break

    for p in papers:
        year = p.get('year')
        if year and (year < int(DATE_FROM) or year > int(DATE_TO)):
            continue

        doi = (p.get('externalIds') or {}).get('DOI', '')
        authors = '; '.join(
            a.get('name', '') for a in (p.get('authors') or [])[:6]
        )
        s2_records.append({
            'record_id': str(uuid.uuid4()),
            'source_db': 'SemanticScholar',
            'title': p.get('title') or '',
            'authors': authors,
            'year': year,
            'journal_venue': p.get('venue') or '',
            'abstract': p.get('abstract') or '',
            'doi': doi,
            'url': f'https://www.semanticscholar.org/paper/{p["paperId"]}',
        })

    total = data.get('total', 0)
    print(f'  Offset {offset}: got {len(papers)} (total: {total})')

    if offset + limit >= min(total, MAX_RESULTS_PER_SOURCE):
        break
    offset += limit
    time.sleep(1.5)

if s2_records:
    out_path = 'data/raw/semanticscholar_results.json'
    with open(out_path, 'w') as f:
        json.dump(s2_records, f, indent=2)
    print(f'  ✓ Saved {len(s2_records)} records → {out_path}')

print('\n── Automated search summary ──────────────────────')
for fname in ['pubmed_results.xml','arxiv_results.json',
              'openalex_results.json','semanticscholar_results.json']:
    p = f'data/raw/{fname}'
    if os.path.exists(p):
        size = os.path.getsize(p)
        print(f'  ✓ {fname:45s} {size:>8,} bytes')
    else:
        print(f'  ✗ {fname} — not generated')

## Cell 3b — Manual exports (institutional databases)

The databases below require university/library login. Run these searches yourself,
then upload the exported files using the upload widget below.

---

### PubMed (manual export — NBIB format)
1. Run your search at [pubmed.ncbi.nlm.nih.gov](https://pubmed.ncbi.nlm.nih.gov)
2. Use the refined strategy saved in `docs/search_strategies.md`
3. Select all results → **Send to** → **Citation manager** → **Create file**
4. Save as `pubmed.nbib`  *(the pipeline reads NBIB natively)*

**Refined PubMed strategy:**
```
(
  ("large language model"[Title/Abstract] OR "large language models"[Title/Abstract]
   OR ChatGPT[Title/Abstract] OR GPT-4[Title/Abstract] OR "GPT 4"[Title/Abstract]
   OR Claude[Title/Abstract] OR Llama[Title/Abstract] OR Gemini[Title/Abstract])
  AND
  ("qualitative analysis"[Title/Abstract] OR "qualitative data analysis"[Title/Abstract]
   OR "thematic analysis"[Title/Abstract] OR "content analysis"[Title/Abstract]
   OR "grounded theory"[Title/Abstract] OR coding[Title/Abstract]
   OR codebook[Title/Abstract] OR "theme development"[Title/Abstract]
   OR "automated coding"[Title/Abstract])
)
AND ("2023/01/01"[Date - Publication] : "2026/12/31"[Date - Publication])
AND (english[Language])
```

---

### Scopus
1. Go to [scopus.com](https://www.scopus.com) (requires institutional login)
2. Click **Advanced search** → paste strategy from `docs/search_strategies.md`
3. Filter: **Date 2023–2026** · **Language: English**
4. Select all → **Export** → **RIS** → include Abstract + Keywords
5. Save as `scopus.ris`

### Web of Science
1. Go to [webofscience.com](https://www.webofscience.com) → **Advanced Search**
2. Paste strategy, set **Timespan: 2023–2026**, **Language: English**
3. Select all → **Export** → **RIS** → include all fields
4. Save as `wos.ris`

### PsycINFO (via EBSCOhost)
1. Access via your library → EBSCOhost → PsycINFO database
2. **Advanced Search** → paste query
3. Limiter: **Publication Date 2023–2026** · **Language: English** · **Peer Reviewed**
4. Select all → **Export** → **Direct Export to RIS**
5. Save as `psycinfo.ris`

### ACM Digital Library
1. Go to [dl.acm.org](https://dl.acm.org) → **Advanced Search**
2. Filter: **Published 2023–2026**
3. Export → **BibTeX** or **CSV**
4. Save as `acm.bib` or `acm.csv`

### IEEE Xplore
1. Go to [ieeexplore.ieee.org](https://ieeexplore.ieee.org) → **Advanced Search**
2. Paste query, set **Year: 2023–2026**
3. Export → **CSV** → include Abstract
4. Save as `ieee.csv`


### Upload manual export files

In [ ]:
from google.colab import files as colab_files

SUPPORTED = ('.nbib', '.ris', '.csv', '.bib', '.json', '.xml', '.txt')
print('Upload your exported files:')
print('  • PubMed  → pubmed.nbib  (Citation manager export)')
print('  • Scopus  → scopus.ris')
print('  • WoS     → wos.ris')
print('  • Others  → psycinfo.ris, acm.bib, ieee.csv, ...')
print('\nPress Cancel if you have no files to upload right now.\n')

try:
    uploaded = colab_files.upload()
    for fname, content in uploaded.items():
        if any(fname.lower().endswith(ext) for ext in SUPPORTED):
            dest = f'data/raw/{fname}'
            with open(dest, 'wb') as f:
                f.write(content)
            fmt = fname.rsplit('.', 1)[-1].upper()
            print(f'  ✓ {fname} ({len(content):,} bytes) → data/raw/  [{fmt}]')
        else:
            print(f'  ✗ Skipped (unsupported format): {fname}')
except Exception:
    print('No files uploaded — you can re-run this cell later.')

print('\nAll files in data/raw/:')
!ls -lh data/raw/ 2>/dev/null || echo '  (empty)'

## Cell 4 — (Optional) Upload PDFs for full-text screening

Name each file `<record_id>.pdf`. You can also do this later via the UI.

In [ ]:
from google.colab import files as colab_files
print('Select PDF files (Cancel to skip):')
try:
    uploaded = colab_files.upload()
    for fname, content in uploaded.items():
        if fname.lower().endswith('.pdf'):
            dest = f'data/pdfs/{fname}'
            with open(dest, 'wb') as f:
                f.write(content)
            print(f'  ✓ {fname}')
        else:
            print(f'  ✗ Skipped (not PDF): {fname}')
except Exception:
    print('No PDFs uploaded — add them later via the UI.')

## Cell 5 — Start server + open public URL

Uses Cloudflare Tunnel — no account, no token, free.

In [ ]:
import subprocess, time, re, sys, os, urllib.request
from IPython.display import display, HTML

APP_DIR = '/content/sysrev/systematic-review'
PORT = 8000

os.system(f'fuser -k {PORT}/tcp 2>/dev/null; pkill cloudflared 2>/dev/null; true')
time.sleep(1)

if not os.path.exists('/content/cloudflared'):
    print('Downloading cloudflared...')
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
         -O /content/cloudflared && chmod +x /content/cloudflared

server = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'api.main:app',
     '--host', '0.0.0.0', '--port', str(PORT), '--log-level', 'warning'],
    cwd=APP_DIR, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)

print('Starting server', end='')
for _ in range(30):
    time.sleep(1)
    try:
        urllib.request.urlopen(f'http://localhost:{PORT}/api/pipeline/status', timeout=2)
        print(' ready.')
        break
    except Exception:
        print('.', end='', flush=True)
else:
    print('\nERROR: Server did not start. Re-run this cell.')

tunnel = subprocess.Popen(
    ['/content/cloudflared', 'tunnel', '--url', f'http://localhost:{PORT}'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)

public_url = None
print('Opening Cloudflare tunnel', end='')
for _ in range(40):
    line = tunnel.stdout.readline().decode('utf-8', errors='ignore')
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        print(' done.')
        break
    print('.', end='', flush=True)

if public_url:
    app_url = f'{public_url}/app'
    print(f'\n  UI:  {app_url}')
    print(f'  API: {public_url}/docs')
    display(HTML(f'''
    <div style="margin:12px 0;padding:18px;background:#0f1117;border:2px solid #6c8eff;
      border-radius:10px;font-family:monospace;">
      <div style="color:#8890aa;font-size:11px;margin-bottom:6px;">OPEN IN YOUR BROWSER</div>
      <a href="{app_url}" target="_blank" rel="noopener"
        style="color:#6c8eff;font-size:18px;font-weight:bold;text-decoration:none;">{app_url}</a>
      <div style="color:#4caf81;font-size:12px;margin-top:10px;">
        ✓ API key from Colab Secrets &nbsp;|&nbsp; ✓ No account needed &nbsp;|&nbsp; ✓ HTTPS
      </div>
    </div>
    '''))
else:
    print('\nERROR: Could not get tunnel URL. Re-run this cell.')

## Cell 6 — Run pipeline stages

Run from here, or use the **Run Pipeline** panel in the UI.

In [ ]:
import requests, time

PORT = 8000
BASE = f'http://localhost:{PORT}/api'

def _server_ok():
    try:
        requests.get(f'{BASE}/pipeline/status', timeout=3)
        return True
    except Exception:
        print('╔══════════════════════════════════════════════════════════╗')
        print('║  SERVER NOT RUNNING — please run Cell 5 first, then      ║')
        print('║  re-run this cell and try again.                         ║')
        print('╚══════════════════════════════════════════════════════════╝')
        return False

def run_stage(stage, limit=None):
    """Run a pipeline stage.
    import / dedup / reset_screening → synchronous, returns immediately.
    title_screening / fulltext_screening / extraction → background task;
                      call status() to monitor progress.
    """
    if not _server_ok(): return
    r = requests.post(f'{BASE}/pipeline/run', json={'stage': stage, 'limit': limit}, timeout=300)
    r.raise_for_status()
    resp = r.json()

    if resp.get('status') == 'completed':
        print(f'✓ {stage} complete.')
        if 'stats' in resp:
            st = resp['stats']
            if stage == 'import':
                print(f'  Imported: {st.get("imported","?")} records')
                for fname, n in (st.get('per_file') or {}).items():
                    print(f'    {str(n):>5}  {fname}')
            elif stage == 'dedup':
                print(f'  Input: {st.get("total_input","?")}  |  Unique: {st.get("unique_records","?")}')
                print(f'  DOI dupes: {st.get("duplicates_by_doi","?")}  '
                      f'Title dupes: {st.get("duplicates_by_exact_title","?")}  '
                      f'Fuzzy dupes: {st.get("duplicates_by_fuzzy_title","?")}')
            elif stage == 'reset_screening':
                print(f'  Rolled back: {st.get("rolled_back","?")} records → DEDUP stage')
                print(f'  Human-verified decisions preserved: {st.get("human_verified_kept","?")}')
        status()
    else:
        print(f'⏳ {stage} started in background.')
        print('  Call status() to monitor. This can take minutes to hours.')

def _print_status(data):
    c = data['prisma_counts']
    log = data.get('stage_log', {})
    running = data.get('running_stage', '')

    if running:
        print(f'\n  ⏳  STAGE RUNNING: {running}  — call status() again in ~30s\n')

    print('── PRISMA Flow ───────────────────────────────────────────────')
    print(f'  Identified (all imports)            {c.get("identified",0):>6}')
    print(f'  Duplicates removed                  {c.get("duplicates_removed",0):>6}')
    print(f'  After dedup (unique)                {c.get("after_dedup",0):>6}')
    ats = c.get("awaiting_title_screening", 0)
    hint = '  ← run_stage("title_screening") next' if ats > 0 and not running else ''
    print(f'  Awaiting title screening            {ats:>6}{hint}')
    print(f'  Title/abstract excluded             {c.get("title_abstract_excluded",0):>6}')
    print(f'  Title/abstract included             {c.get("title_abstract_included",0):>6}')
    print(f'  Full text needed (upload PDFs)      {c.get("full_text_needed",0):>6}')
    print(f'  Full text excluded                  {c.get("full_text_excluded",0):>6}')
    print(f'  Final included                      {c.get("final_included",0):>6}')
    nhv = c.get('needs_human_verification', 0)
    mark = '  ⚠ ← review in UI' if nhv else ''
    print(f'  Needs human verification            {nhv:>6}{mark}')
    print()

def status():
    if not _server_ok(): return
    _print_status(requests.get(f'{BASE}/pipeline/status').json())

def uncertain():
    if not _server_ok(): return
    data = requests.get(f'{BASE}/records/uncertain/list').json()
    print(f'\nNeeds Human Verification: {data["count"]} records')
    for r in data['records'][:10]:
        print(f'  [{r["record_id"][:8]}] {r["title"][:70]}')

def fulltext_needed():
    if not _server_ok(): return
    data = requests.get(f'{BASE}/records/fulltext-needed/list').json()
    print(f'\nFull Text Needed: {data["count"]} papers')
    for r in data['records'][:10]:
        print(f'  {r["doi"] or "no-doi":40s}  {r["title"][:50]}')

print('Helpers ready.')
print('  run_stage("import")          — import files (instant result)')
print('  run_stage("dedup")           — deduplicate (instant result + counts)')
print('  run_stage("reset_screening") — roll back all non-verified screening results')
print('  run_stage("title_screening") — start AI screening (background)')
print('  status()                     — PRISMA flow counts')

### Cell 6a — Diagnostics: check what files are in data/raw

Run this before import to verify all your files are present and readable.
If the count is lower than expected, see the encoding fix notes below.

In [ ]:
import os, sys, subprocess

# ── Ensure rispy is available (safe to run even if already installed) ────
try:
    import rispy
except ImportError:
    print('Installing rispy...', end='')
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'rispy', '-q'], check=True)
    import rispy
    print(' done.')

RAW = 'data/raw'
SUPPORTED = {'.nbib', '.ris', '.csv', '.bib', '.json', '.xml', '.txt'}

if not os.path.exists(RAW):
    print('data/raw does not exist — run Cell 1 first (or Cell 8 to restore from Drive).')
else:
    files = sorted(f for f in os.listdir(RAW) if os.path.splitext(f)[1].lower() in SUPPORTED)
    other = sorted(f for f in os.listdir(RAW) if os.path.splitext(f)[1].lower() not in SUPPORTED and not f.startswith('.'))

    if not files and not other:
        print('data/raw/ is EMPTY.')
        print('→ Run Cell 3a for automated searches, or Cell 3b to upload your export files.')
    else:
        print(f'Supported files in data/raw/ ({len(files)} files):')
        total_bytes = 0
        for fname in files:
            size = os.path.getsize(f'{RAW}/{fname}')
            total_bytes += size
            ext = os.path.splitext(fname)[1].upper()
            print(f'  {ext:6s}  {size:>10,} bytes  {fname}')
        print(f'\n  Total: {total_bytes:,} bytes across {len(files)} files')
        if other:
            print(f'\nUnsupported files (will be skipped): {other}')

    # Quick parse-test: count records per file before import
    print('\n── Quick record count (before import/dedup) ────────────────')
    sys.path.insert(0, '.')
    try:
        # Reload importer to pick up rispy now that it's installed
        import importlib
        if 'pipeline.importer' in sys.modules:
            importlib.reload(sys.modules['pipeline.importer'])
        from pipeline.importer import (
            import_ris, import_nbib, import_csv, import_pubmed_xml, import_json
        )
        parsers = {
            '.nbib': import_nbib, '.txt': import_nbib,
            '.ris':  lambda p: import_ris(p, source_db='check'),
            '.csv':  lambda p: import_csv(p, source_db='check'),
            '.xml':  import_pubmed_xml,
            '.json': lambda p: import_json(p, source_db='check'),
        }
        grand_total = 0
        for fname in files:
            ext = os.path.splitext(fname)[1].lower()
            path = f'{RAW}/{fname}'
            if ext in parsers:
                try:
                    recs = parsers[ext](path)
                    n = len(recs)
                    grand_total += n
                    print(f'  {n:>5} records  {fname}')
                except Exception as e:
                    print(f'  ERROR  {fname}: {e}')
        print(f'\n  Grand total (before dedup): {grand_total} records')
        if grand_total > 0:
            print('\n  ✓ Files look good — run Cell 6 → run_stage("import") to import them.')
    except ImportError as e:
        print(f'  Could not import parsers: {e}')
        print('  Make sure Cell 1 has been run in this session.')

### Cell 6b — Debug: verify API key + check agent errors

Run this if title screening is stuck or all records show "Needs Human Verification".

In [ ]:
import requests

PORT = 8000
BASE = f'http://localhost:{PORT}/api'

# ── Step 1: Test that your API key works ─────────────────────────────────
print('Testing OpenAI API key...')
try:
    r = requests.get(f'{BASE}/debug/test-api-key', timeout=30)
    result = r.json()
    if result.get('status') == 'ok':
        print(f'  ✓ API key OK — model {result["model"]} responded:')
        print(f'    {result["response"]}')
    else:
        print(f'  ✗ API ERROR: {result["error_type"]}: {result["error_message"]}')
        print(f'    Hint: {result.get("hint", "")}')
        if 'Authentication' in result.get('error_type', '') or 'Invalid' in result.get('error_message', ''):
            print()
            print('  ── How to fix ─────────────────────────────────────────')
            print('  1. Go to Cell 2 and verify your OPENAI_API_KEY secret')
            print('  2. Re-run Cell 2 to rewrite .env')
            print('  3. Re-run Cell 5 to restart the server')
            print('  4. Re-run this cell to verify the fix')
except Exception as e:
    print(f'  Cannot reach server: {e}')
    print('  → Run Cell 5 first')

print()

# ── Step 2: Show recent agent screening errors ────────────────────────────
print('Recent agent errors (last 20):')
try:
    r2 = requests.get(f'{BASE}/debug/agent-errors', timeout=10)
    err_data = r2.json()
    if err_data['error_count'] == 0:
        print('  (none — agents running without errors)')
    else:
        print(f'  Total: {err_data["error_count"]} errors')
        for msg in err_data['errors'][-20:]:
            print(f'  {msg}')
except Exception as e:
    print(f'  Cannot reach server: {e}')

### Cell 6c — Auto-save to Drive during long runs

Run this **before** starting title_screening. It saves `pipeline_state.jsonl`
to Google Drive every 10 minutes in the background so a Colab timeout won't
lose your screening progress.

In [ ]:
import threading, shutil, os, time
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

DRIVE     = '/content/drive/MyDrive/SysRev_LLMs_QDA'
STATE_SRC = '/content/sysrev/systematic-review/data/pipeline_state.jsonl'
INTERVAL  = 10 * 60   # seconds between saves (10 minutes)

os.makedirs(DRIVE, exist_ok=True)

_autosave_stop = threading.Event()

def _autosave_loop():
    save_count = 0
    while not _autosave_stop.wait(timeout=INTERVAL):
        if os.path.exists(STATE_SRC):
            dest = f'{DRIVE}/pipeline_state.jsonl'
            shutil.copy(STATE_SRC, dest)
            size = os.path.getsize(dest)
            save_count += 1
            print(f'[auto-save #{save_count}] pipeline_state.jsonl → Drive  ({size:,} bytes,  {time.strftime("%H:%M:%S")})')

_autosave_thread = threading.Thread(target=_autosave_loop, daemon=True)
_autosave_thread.start()

print(f'✓ Auto-save started — saving to Drive every {INTERVAL//60} minutes.')
print(f'  File: {DRIVE}/pipeline_state.jsonl')
print()
print('  If Colab resets mid-screening:')
print('    Cell 1 → Cell 2 → Cell 8 → Cell 5 → Cell 6')
print('    Then run_stage("title_screening") to resume from where it left off.')
print()
print('  To stop auto-save: _autosave_stop.set()')

In [ ]:
# Uncomment and run one at a time. Check status() between stages.

# run_stage('import')          # load all files from data/raw/
# status()

# run_stage('dedup')           # remove duplicates
# status()

# run_stage('title_screening') # two-agent title/abstract screening
# status()
# uncertain()                  # ← review these in the UI before continuing

# run_stage('fulltext_screening')
# fulltext_needed()            # ← upload those PDFs, then re-run

# run_stage('extraction')
# status()

## Cell 7 — Save to Google Drive (run before closing)

In [ ]:
from google.colab import drive
import shutil, os

drive.mount('/content/drive', force_remount=False)
DRIVE = '/content/drive/MyDrive/SysRev_LLMs_QDA'
APP   = '/content/sysrev/systematic-review'
os.makedirs(DRIVE, exist_ok=True)

for src, label in [
    (f'{APP}/data/pipeline_state.jsonl', 'pipeline_state.jsonl'),
    (f'{APP}/data/output', 'output/'),
    (f'{APP}/data/raw',    'raw/'),
    (f'{APP}/data/pdfs',   'pdfs/'),
]:
    if not os.path.exists(src):
        continue
    if os.path.isfile(src):
        shutil.copy(src, f'{DRIVE}/{label}')
    else:
        if os.listdir(src):
            shutil.copytree(src, f'{DRIVE}/{label.rstrip("/")}', dirs_exist_ok=True)
    print(f'  ✓ Saved: {label}')

print(f'\nSaved to Google Drive: {DRIVE}')

## Cell 8 — Restore after session reset

Run: **Cell 1 → Cell 2 → this cell → Cell 5** to resume.

In [ ]:
from google.colab import drive
import shutil, os

drive.mount('/content/drive', force_remount=False)
DRIVE = '/content/drive/MyDrive/SysRev_LLMs_QDA'
APP   = '/content/sysrev/systematic-review'

if not os.path.exists(DRIVE):
    print('No saved session in Drive — starting fresh.')
else:
    restored = []
    for src_name, dest_path, is_file in [
        ('pipeline_state.jsonl', f'{APP}/data/pipeline_state.jsonl', True),
        ('output',               f'{APP}/data/output',               False),
        ('raw',                  f'{APP}/data/raw',                  False),
        ('pdfs',                 f'{APP}/data/pdfs',                 False),
    ]:
        src = f'{DRIVE}/{src_name}'
        if not os.path.exists(src):
            continue
        os.makedirs(os.path.dirname(dest_path), exist_ok=True)
        if is_file:
            shutil.copy(src, dest_path)
        else:
            shutil.copytree(src, dest_path, dirs_exist_ok=True)
        restored.append(src_name)

    print(f'Restored: {" | ".join(restored) or "nothing found"}')
    print('Now run Cell 5 to restart the server.')

## Cell 9 — Stop the server

In [ ]:
try:
    tunnel.terminate()
    server.terminate()
except Exception:
    pass
os.system('pkill cloudflared 2>/dev/null; true')
print('Stopped.')

---
## Summary

| Database | How searched | Format saved |
|----------|-------------|-------------|
| **PubMed** | Cell 3a (auto, free API) | `pubmed_results.xml` |
| **arXiv** | Cell 3a (auto, free API) | `arxiv_results.json` |
| **OpenAlex** | Cell 3a (auto, free API — covers WoS/Scopus) | `openalex_results.json` |
| **Semantic Scholar** | Cell 3a (auto, free API) | `semanticscholar_results.json` |
| **Scopus** | Cell 3b (manual export, needs login) | `scopus.ris` |
| **Web of Science** | Cell 3b (manual export, needs login) | `wos.ris` |
| **PsycINFO** | Cell 3b (manual export, needs login) | `psycinfo.ris` |
| **ACM DL** | Cell 3b (manual export, needs login) | `acm.bib` |
| **IEEE Xplore** | Cell 3b (manual export, needs login) | `ieee.csv` |

All files land in `data/raw/`. The import stage handles deduplication across sources.
